# Cyclistic Bike Usage Analysis — January to April 2024

This notebook documents a comparative analysis of Cyclistic bike-share trip data from January through April 2024, examining behavioral differences between casual riders and annual members to guide the company's membership conversion strategy. The original analysis was completed in Google Sheets; this notebook replicates and extends that work using Python and the pandas library, making the methodology transparent and reproducible.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import datetime

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

## Part 1: Data Collection

The data comes from Cyclistic's internal trip records, covering January through April 2024. Each month's data is stored in a separate CSV file provided by Divvy, the underlying bike-share operator. The four files are loaded individually and concatenated into a single DataFrame for unified analysis.

The columns used in this analysis are:
- `ride_id` — unique identifier for each trip
- `rideable_type` — the type of bike used (`classic_bike` or `electric_bike`)
- `started_at` — timestamp when the trip began
- `ended_at` — timestamp when the trip ended
- `member_casual` — user classification (`member` or `casual`)

In [ ]:
jan = pd.read_csv('202401-divvy-tripdata.csv')
feb = pd.read_csv('202402-divvy-tripdata.csv')
mar = pd.read_csv('202403-divvy-tripdata.csv')
apr = pd.read_csv('202404-divvy-tripdata.csv')

data = pd.concat([jan, feb, mar, apr], ignore_index=True)

data.head(10)

The combined dataset contains records across four months. Each row represents a single trip. The head() preview confirms the expected column structure and shows a mix of electric and classic bike trips from both casual and member riders.

In [ ]:
data.info()
print('\nDataset dimensions:', data.shape)

---

## Part 2: Data Cleaning and Transformation

Before analysis, the raw data requires several transformations. The timestamp columns (`started_at`, `ended_at`) are read as strings and must be converted to datetime objects. From these, we derive three new columns:
- `ride_length` — trip duration in minutes, computed as the difference between end and start times
- `day_of_week` — the numeric weekday of the trip start (Monday = 0, Sunday = 6, per pandas convention)
- `month` — the calendar month name for grouping and chart labels

Invalid rows (zero or negative duration, trips exceeding 24 hours, and duplicate ride IDs) are removed to ensure analytical integrity.

In [ ]:
data['started_at'] = pd.to_datetime(data['started_at'])
data['ended_at'] = pd.to_datetime(data['ended_at'])

data['ride_length'] = (data['ended_at'] - data['started_at']).dt.total_seconds() / 60

# pandas dayofweek: Monday = 0, Sunday = 6
data['day_of_week'] = data['started_at'].dt.dayofweek

data['month'] = data['started_at'].dt.month_name()
data['month_num'] = data['started_at'].dt.month

In [ ]:
print('Shape before cleaning:', data.shape)

# Remove trips with invalid durations
data = data[(data['ride_length'] > 0) & (data['ride_length'] <= 1440)]

# Remove duplicate ride IDs
data = data.drop_duplicates(subset='ride_id')

print('Shape after cleaning: ', data.shape)

Rows with `ride_length` at or below zero represent data entry errors or docking mistakes and cannot represent real trips. Rows exceeding 1,440 minutes (24 hours) are treated as outliers, likely indicating bikes that were not properly returned. Duplicate `ride_id` values are removed to prevent double-counting. The printed shapes show how many records were removed at this stage.

In [ ]:
data.describe()[['ride_length']]

The summary statistics for `ride_length` confirm that the cleaning step removed the erroneous values. The mean and median give a sense of typical trip duration, while the max confirms no trips exceed 24 hours.

---

## Part 3: Ride Distribution Overview

Before diving into breakdowns by bike type and day of week, we examine how rides are distributed between casual riders and annual members, both in total and by month. This establishes the baseline context — members are expected to dominate volume, but we want to quantify the gap and observe any monthly trends.

In [ ]:
# Total rides by user type
total_by_type = data.groupby('member_casual')['ride_id'].count().reset_index()
total_by_type.columns = ['User Type', 'Total Rides']
total_by_type['Share (%)'] = (total_by_type['Total Rides'] / total_by_type['Total Rides'].sum() * 100).round(1)
print(total_by_type)

# Monthly breakdown
monthly_share = data.groupby(['month_num', 'month', 'member_casual'])['ride_id'].count().reset_index()
monthly_share.columns = ['Month Num', 'Month', 'User Type', 'Rides']
monthly_share = monthly_share.sort_values('Month Num')
print('\n', monthly_share)

Members account for over 75% of rides in January and February. Casual riders remain below 30% across all months, though their share grows incrementally each month — suggesting seasonal uptick in casual usage as winter gives way to spring.

---

## Part 4: Ride Length by Bike Type

This section examines how long, on average, each user type rides each bike category (classic vs electric). Differences in ride duration by bike type reveal usage intensity patterns that go beyond simple trip counts. If casual riders consistently take longer rides on both bike types, it suggests they are using the service for leisure or exploration rather than routine commuting.

In [ ]:
ride_len_bike = data.groupby(['month_num', 'month', 'member_casual', 'rideable_type'])['ride_length'].mean().reset_index()
ride_len_bike.columns = ['Month Num', 'Month', 'User Type', 'Bike Type', 'Avg Ride Length (min)']
ride_len_bike['Avg Ride Length (min)'] = ride_len_bike['Avg Ride Length (min)'].round(2)
ride_len_bike = ride_len_bike.sort_values('Month Num')
print(ride_len_bike)

The table shows average ride length in minutes for each combination of month, user type, and bike type. Look for consistent patterns across months rather than single-month anomalies.

In [ ]:
months_order = ['January', 'February', 'March', 'April']

fig, axes = plt.subplots(2, 2, figsize=(14, 10), sharey=False)
axes = axes.flatten()

for idx, month in enumerate(months_order):
    df_m = ride_len_bike[ride_len_bike['Month'] == month]
    sns.barplot(data=df_m, x='Bike Type', y='Avg Ride Length (min)', hue='User Type', ax=axes[idx], palette='muted')
    axes[idx].set_title(month)
    axes[idx].set_xlabel('Bike Type')
    axes[idx].set_ylabel('Avg Ride Length (min)')

plt.suptitle('Average Ride Length by Bike Type and User Type', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

Casual riders ride both bike types for longer durations than members across all four months. Classic bikes consistently yield longer average rides for both user types, likely because electric-assist encourages faster, shorter trips. These patterns hold in every month without exception, reinforcing that the casual vs member behavioral difference is structural, not seasonal.

---

## Part 5: Ride Length by Day of Week

Analyzing average ride duration by day of the week shows when each user group takes longer, more leisure-style trips versus shorter commute-like trips. If members are primarily commuters, their ride lengths should be relatively flat across weekdays. Casual riders, who likely use the service recreationally, should show longer rides on weekends.

In [ ]:
day_map = {0: 'Monday', 1: 'Tuesday', 2: 'Wednesday', 3: 'Thursday', 4: 'Friday', 5: 'Saturday', 6: 'Sunday'}
data['weekday_name'] = data['day_of_week'].map(day_map)

ride_len_day = data.groupby(['month_num', 'month', 'member_casual', 'day_of_week', 'weekday_name'])['ride_length'].mean().reset_index()
ride_len_day.columns = ['Month Num', 'Month', 'User Type', 'Day Num', 'Weekday', 'Avg Ride Length (min)']
ride_len_day['Avg Ride Length (min)'] = ride_len_day['Avg Ride Length (min)'].round(2)
ride_len_day = ride_len_day.sort_values(['Month Num', 'Day Num'])

In [ ]:
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for idx, month in enumerate(months_order):
    df_m = ride_len_day[ride_len_day['Month'] == month]
    sns.lineplot(data=df_m, x='Weekday', y='Avg Ride Length (min)', hue='User Type', marker='o', ax=axes[idx], palette='muted')
    axes[idx].set_title(month)
    axes[idx].set_xticks(range(7))
    axes[idx].set_xticklabels(day_order, rotation=30)

plt.suptitle('Average Ride Length by Day of Week and User Type', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

Casual riders average longer rides than members on every weekday, in every month. In March and April, both user types peak on Sundays, consistent with weekend leisure riding. In January, casual riders peaked on Monday while members peaked on Wednesday. Mid-week days consistently show the shortest average rides for both groups, pointing toward routine, efficient commute-style usage during those days.

---

## Part 6: Number of Rides by Day of Week

Ride volume by weekday reveals whether usage patterns align with commuting or leisure. Members — who likely use the service to commute — would be expected to show mid-week peaks. Casual riders, who tend toward recreational use, would concentrate rides on weekends. This section quantifies those differences and tracks whether they hold across all four months.

In [ ]:
rides_by_day = data.groupby(['month_num', 'month', 'member_casual', 'day_of_week', 'weekday_name'])['ride_id'].count().reset_index()
rides_by_day.columns = ['Month Num', 'Month', 'User Type', 'Day Num', 'Weekday', 'Ride Count']
rides_by_day = rides_by_day.sort_values(['Month Num', 'Day Num'])

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for idx, month in enumerate(months_order):
    df_m = rides_by_day[rides_by_day['Month'] == month]
    sns.barplot(data=df_m, x='Weekday', y='Ride Count', hue='User Type', ax=axes[idx], palette='muted', order=day_order)
    axes[idx].set_title(month)
    axes[idx].set_xlabel('Day of Week')
    axes[idx].set_ylabel('Number of Rides')
    axes[idx].tick_params(axis='x', rotation=30)

plt.suptitle('Number of Rides by Day of Week and User Type', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

Members make more rides than casual riders on every day of the week, across all four months. Members peak mid-week: Wednesdays stand out in March and April. Casual riders peak on Saturdays in March and April. In January and February, both groups concentrate rides mid-week (Wednesday through Thursday), likely reflecting winter commuting patterns when leisure cycling is less appealing.

---

## Part 7: Monthly Ride Share by User Type

Looking at the proportion of rides each user type contributes per month quantifies the membership gap and tracks whether casual usage grows as the season progresses into spring. A growing casual share month-over-month would signal increasing opportunity for conversion campaigns timed to seasonal demand.

In [ ]:
monthly_totals = data.groupby(['month_num', 'month', 'member_casual'])['ride_id'].count().reset_index()
monthly_totals.columns = ['Month Num', 'Month', 'User Type', 'Rides']
monthly_totals['Total'] = monthly_totals.groupby('Month')['Rides'].transform('sum')
monthly_totals['Share (%)'] = (monthly_totals['Rides'] / monthly_totals['Total'] * 100).round(1)
monthly_totals = monthly_totals.sort_values('Month Num')
print(monthly_totals[['Month', 'User Type', 'Rides', 'Share (%)']])

The table confirms the membership dominance in raw numbers and percentage terms. The Share (%) column is the key figure for tracking the casual-to-member gap across months.

In [ ]:
pivot = monthly_totals.pivot_table(index='Month', columns='User Type', values='Share (%)', aggfunc='sum')
pivot = pivot.reindex(months_order)

pivot.plot(kind='bar', stacked=True, colormap='Set2', figsize=(10, 6))
plt.title('Monthly Ride Share by User Type (%)')
plt.xlabel('Month')
plt.ylabel('Share of Total Rides (%)')
plt.xticks(rotation=0)
plt.legend(title='User Type')
plt.tight_layout()
plt.show()

Annual members dominate ride volume at over 75% in January and February. Casual riders account for less than 30% across all months, though their share increases each successive month from January through April — suggesting seasonal growth in casual usage as weather improves. This trend represents a window of opportunity: casual riders are entering the network and are potentially most receptive to membership messaging in March and April.

---

## Part 8: Number of Rides by Bike Type

Bike type preferences reveal which hardware each user group gravitates toward. Understanding whether casual riders favor electric bikes and members favor classic bikes informs both fleet allocation decisions and promotional targeting — for example, electric bike promotions may resonate more strongly with casual users.

In [ ]:
rides_bike = data.groupby(['month_num', 'month', 'member_casual', 'rideable_type'])['ride_id'].count().reset_index()
rides_bike.columns = ['Month Num', 'Month', 'User Type', 'Bike Type', 'Ride Count']
rides_bike = rides_bike.sort_values(['Month Num', 'User Type'])
print(rides_bike)

The table breaks down ride counts by month, user type, and bike type. Look for month-to-month shifts in bike preference, particularly the transition from classic to electric as spring progresses.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for idx, month in enumerate(months_order):
    df_m = rides_bike[rides_bike['Month'] == month]
    sns.barplot(data=df_m, x='Bike Type', y='Ride Count', hue='User Type', ax=axes[idx], palette='muted')
    axes[idx].set_title(month)
    axes[idx].set_xlabel('Bike Type')
    axes[idx].set_ylabel('Number of Rides')

plt.suptitle('Number of Rides by Bike Type and User Type', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

Members ride significantly more than casual riders in both bike categories across all months. Members preferred classic bikes in January and February, then shifted toward electric bikes in March and April. Casual riders preferred electric bikes in January, March, and April, but chose classic bikes in February — a divergence worth monitoring in future months to confirm whether it is a consistent pattern.

---

## Conclusions and Recommendations

This section summarizes the core findings from the January–April 2024 analysis and translates them into actionable recommendations for Cyclistic's marketing and operations teams.

### Key Conclusions

1. Casual riders rode both bike types for longer durations than members across all four months, indicating that casual users engage in longer, more exploratory trips regardless of the equipment they choose.
2. Casual riders averaged longer ride times than members on every weekday in every month, suggesting a consistent behavioral difference rooted in purpose-of-use rather than day-specific factors.
3. Members made more rides than casual riders on every day of the week in all four months, demonstrating that higher engagement frequency — not trip length — is the defining characteristic of annual membership.
4. Despite averaging more minutes per ride, casual riders accounted for less than 30% of monthly ride volume in all months, underscoring that frequency of use, not duration, is the primary driver of total system utilization.
5. Casual riders generally preferred electric bikes except in February; members preferred classic bikes in January and February and shifted to electric bikes in March and April — preferences that appear to track seasonal conditions.

### Recommendations

- Target casual riders with weekend promotions and electric bike incentives, aligning marketing timing with their peak days (Saturdays in spring) and their demonstrated preference for electric bikes — offering a tangible, usage-relevant benefit that members already enjoy.
- Introduce usage-based membership prompts for casual riders who complete three or more rides in a single month, leveraging their existing engagement patterns to surface conversion messaging at moments of demonstrated interest.
- Optimize fleet distribution by increasing electric bike availability on weekends to meet casual rider demand, and increasing mid-week electric availability to serve members who are transitioning to electric bikes in spring — ensuring supply aligns with the seasonal demand shifts observed in March and April.